<img src="../assets/rpt_decision_overview.svg" alt="SAP RPT Decision A2A Coordinator 흐름" style="display:block;width:100%;max-width:1200px;margin:0 auto 24px;border-radius:12px;">


<img src="../../4_agent_patterns/assets/a2a_orchestration_architecture_highlight_decision_one_a2a.png" alt="Agent Harness A2A Orchestrator architecture with the lesson-relevant area highlighted" style="display:block; width:100%; max-width:1400px; margin:18px auto 26px;">


![A2A Coordinator Handoff Workflow](../assets/workflow_diagrams/rpt_a2a_handoff.svg)


| 판단 항목 | Coordinator가 하는 일 |
| --- | --- |
| `next_agent` | 현재 상태에서 다음으로 조사·처리할 Agent를 고릅니다. |
| `handoff_mode` | 즉시 위임, 필수 근거 요청 후 위임, 사람 승인 요청을 고릅니다. |
| `risk` | 생산·계약·비용에 미치는 운영 위험을 분류합니다. |
| `priority` | 낮을수록 대기 가능, 높을수록 오늘 처리해야 함을 뜻합니다. |

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">A2A Coordinator UI</h2>
<p style="margin: 10px 0 0;">참고: <a href="https://rpt.cloud.sap/decisions" target="_blank" rel="noopener noreferrer">play ground</a></p>

</div>
</div>

In [21]:
from pathlib import Path
import importlib
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]

import src.rpt_decision.lab_core as rpt_decision_core
import src.rpt_decision.ui as rpt_decision_ui

rpt_decision_core = importlib.reload(rpt_decision_core)
rpt_decision_ui = importlib.reload(rpt_decision_ui)
rpt_decision_app = rpt_decision_ui.build_rpt_decision_ui()
rpt_decision_app.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', height=760)

* Running on local URL:  http://127.0.0.1:7885
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 28px 0 16px; display: flex; align-items: center; gap: 10px; color: #5f6368; font-size: .9em; font-weight: 700;">
  <span style="height: 1px; flex: 1; background: #c7d2e1;"></span>
  <span>🧩 아래부터 코드 베이스 실습</span>
  <span style="height: 1px; flex: 1; background: #c7d2e1;"></span>
</div>


In [10]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]

from src.rpt_decision.lesson_support.config import load_settings
from src.rpt_decision.lab_core import DEFAULT_DECISION_STATE, run_a2a_coordinator_decision

load_settings()
api_key = os.getenv('SAP_RPT_API_KEY', '').strip()
if not api_key:
    raise RuntimeError('SAP_RPT_API_KEY를 프로젝트 루트 .env에 설정하세요.')

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">Step 1. Decision 요청 계약 확인</h2>
</div>
</div>

In [11]:
from src.rpt_decision.lab_core import build_rpt_decision_request

task_state = DEFAULT_DECISION_STATE
request_preview = build_rpt_decision_request(task_state)
question_contract = pd.DataFrame([
    {
        'question': name,
        'type': spec['type'],
        'candidates_or_scale': ', '.join(spec['criteria']) if isinstance(spec['criteria'], dict) else '0, 1, 2',
    }
    for name, spec in request_preview['questions'].items()
])
display(question_contract)
print('Task state:', task_state)

,question,type,candidates_or_scale
0,next_agent,choice,"hr_agent, scm_agent, rnd_agent"
1,handoff_mode,choice,"delegate_now, request_evidence, request_human_..."
2,risk,choice,"low, medium, high"
3,priority,score,"0, 1, 2"


Task state: 주문 PO-1042의 납품 예정일이 지났지만 입고되지 않았습니다. 생산 라인이 오늘 멈출 수 있으니 공급망 현황과 새 납기일을 확인해 주세요.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">Step 2. 상태 변화에 따른 handoff 검증</h2>
</div>
</div>

In [20]:
scenarios = [
    {
        'name': '식별자 누락',
        'state': '납품 지연이 보고되었지만 주문 번호와 shipment ID가 없습니다. 먼저 필요한 정보를 수집해 주세요.',
    },
    {
        'name': '즉시 공급망 조사',
        'state': 'PO-1042의 납품 예정일이 지났고 오늘 생산 라인이 멈출 수 있습니다. 공급망 현황과 새 납기일을 확인해 주세요.',
    },
    {
        'name': '안전 설계 변경 승인',
        'state': '안전 우려가 있는 제품 설계 변경을 출시 전에 검토해야 하며, 오늘 승인권자의 결정이 필요합니다.',
    },
]

scenario_id = 1  # 0, 1, 2 중 하나를 선택
scenario = scenarios[scenario_id]
decision_result = run_a2a_coordinator_decision(scenario['state'], api_key)

actual_result = pd.DataFrame(decision_result['summary'])
display(actual_result)

,question,type,prediction,confidence
0,next_agent,choice,scm_agent,0.67
1,handoff_mode,choice,delegate_now,0.51
2,risk,choice,high,0.68
3,priority,score,2.0,0.68
